In [2]:
# Imports
import sys
import os
sys.path.append(os.path.abspath(".."))

from PythonCode.CreateTeamDataset import ExtractTeamFeatures

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import joblib

from UniversalFunctions import *

In [3]:
# Get the Datafile of the combined AI teams
COMBINED_DATASET_FILE = "../../data/Output/CombinedDataResults/COMBINED_GA_team_dataset.json"
MODEL_RESULT_FILE = "../../data/Output/CombinedDataResults/Combined_LogRegression_Results.json"

# Set the features the logistic model will determine by
DATASET_COlUMNS = [
    "restricted_count",
    "physical_count",
    "special_count",
    "support_count",
    "priority_user_count",
    "protect_count",
    "fake_out_count",
    "redirection_count",
    "pivot_move_count",
    "setup_move_count",
    "spread_move_user_count",
    "status_move_count",
    "weather_setter_count",
    "terrain_setter_count",
    "intimidate_count",
    "has_fake_out",
    "has_tailwind",
    "has_trick_room",
    "has_redirection",
    "has_wide_guard",
    "has_speed_control",
    "has_protect",
    "has_setup",
    "has_intimidate",
    "has_weather",
    "has_terrain"
]

In [4]:
# Load the data and create the training/testing data
data = LoadJSON(COMBINED_DATASET_FILE)
df = pd.DataFrame(data)

x = df[DATASET_COlUMNS]
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# Check the data to make sure its not incomplete
print(df["source"].value_counts())
print(df["label"].value_counts())
print(df.groupby("source")["label"].value_counts())

source
OLLAMA AI    359
ChatGPT      358
Name: count, dtype: int64
label
0    651
1     66
Name: count, dtype: int64
source     label
ChatGPT    0        292
           1         66
OLLAMA AI  0        359
Name: count, dtype: int64


In [5]:
# Start the model
model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(x_train, y_train)

# Get the prediction and accuracy
y_pred = model.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)

# Format results to be printed out to a file
results = {
    "accuracy": float(accuracy),
    "datasetcolumns": DATASET_COlUMNS,
    "coefficients": {
        feature: float(coef) for feature, coef in zip(DATASET_COlUMNS, model.coef_[0])
    },
    "intercept": float(model.intercept_[0]),
    "classificationReport": classification_report(y_test, y_pred, output_dict=True, zero_division=0),
    "confusionMatrix": confusion_matrix(y_test, y_pred).tolist()
}

In [13]:
# Write results out to file and print here the statistics
WriteJSON(MODEL_RESULT_FILE, results)

print(f"Accuracy: {accuracy}")
print(f"Intercept: {model.intercept_[0]}")
print(f"Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"Confusion Matrix: ")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7777777777777778
Intercept: -0.12015228474981252
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.76      0.86       131
           1       0.28      0.92      0.43        13

    accuracy                           0.78       144
   macro avg       0.63      0.84      0.65       144
weighted avg       0.93      0.78      0.82       144

Confusion Matrix: 
[[100  31]
 [  1  12]]


In [ ]:
# Start of LIME, Created a LIME report on the final team for each of the AI teams after the genetic algorithm
from lime.lime_tabular import LimeTabularExplainer

AI_FINAL_BEST_RESULT_FILE = "../../data/Output/AIResults/final_best_GA_result.json"
AI_LIME_RESULT_FILE = "../../data/Output/AIResults/ai_limeExplanation_result.json"

CHATGPT_FINAL_BEST_RESULT_FILE = "../../data/Output/ChatGPTResults/final_best_GA_result.json"
CHATGPT_LIME_RESULT_FILE = "../../data/Output/ChatGPTResults/chatgpt_limeExplanation_result.json"

# Where to output the LIME file
AI_LIME_HTML_FILE = "../../data/Output/AIResults/ai_lime_explanation.html"
CHATGPT_LIME_HTML_FILE = "../../data/Output/ChatGPTResults/chatgpt_lime_explanation.html"

In [ ]:
# Get the final best result of the AI team
finalBestResult = LoadJSON(AI_FINAL_BEST_RESULT_FILE)
finalBestTeam = finalBestResult["team"]

teamFeatures = ExtractTeamFeatures(finalBestTeam)
featureRow = pd.DataFrame([teamFeatures])[DATASET_COlUMNS]

In [ ]:
# Create the Explainer and a function to make the LIME Explanation
explainer = LimeTabularExplainer(
    training_data=x_train.values,
    feature_names=DATASET_COlUMNS,
    class_names=["below_threshold", "above_threshold"],
    mode="classification")

def BuildLIMEResult(finalBestResultFile, limeResultFile, limeHtmlFile, labelName=""):
    finalBestResult = LoadJSON(finalBestResultFile)
    finalBestTeam = finalBestResult["team"]

    teamFeatures = ExtractTeamFeatures(finalBestTeam)
    featureRow = pd.DataFrame([teamFeatures])[DATASET_COlUMNS]

    exp = explainer.explain_instance(
        data_row=featureRow.iloc[0].values,
        predict_fn=lambda x: model.predict_proba(pd.DataFrame(x, columns=DATASET_COlUMNS)),
        num_features=8
    )

    predictedClass = int(model.predict(featureRow)[0])
    predictedProb = model.predict_proba(featureRow)[0].tolist()

    limeResult = {
        "source": labelName,
        "teamScore": finalBestResult.get("score"),
        "predictedClass": predictedClass,
        "predictedProbabilities": {
            "below_threshold": float(predictedProb[0]),
            "above_threshold": float(predictedProb[1])
        },
        "teamFeatures": teamFeatures,
        "limeExplanation": [
            {
                "feature": feature,
                "weight": float(weight)
            }
            for feature, weight in exp.as_list()
        ]
    }

    WriteJSON(limeResultFile, limeResult)
    exp.save_to_file(limeHtmlFile)

    print(f"\n{labelName} Predicted class:", predictedClass)
    print(f"{labelName} Predicted probabilities:", limeResult["predictedProbabilities"])
    print(f"{labelName} LIME explanation saved to:", limeHtmlFile)

    return limeResult

In [ ]:
# Build the Lime result of both the AI and ChatGPT files
aiLimeResult = BuildLIMEResult(
    AI_FINAL_BEST_RESULT_FILE,
    AI_LIME_RESULT_FILE,
    AI_LIME_HTML_FILE,
    labelName="AI"
)

chatgptLimeResult = BuildLIMEResult(
    CHATGPT_FINAL_BEST_RESULT_FILE,
    CHATGPT_LIME_RESULT_FILE,
    CHATGPT_LIME_HTML_FILE,
    labelName="ChatGPT"
)


AI Predicted class: 0
AI Predicted probabilities: {'below_threshold': 0.9999998576438852, 'above_threshold': 1.42356114742805e-07}
AI LIME explanation saved to: ../../data/Output/AIResults/ai_lime_explanation.html

ChatGPT Predicted class: 1
ChatGPT Predicted probabilities: {'below_threshold': 0.19652489329229583, 'above_threshold': 0.8034751067077042}
ChatGPT LIME explanation saved to: ../../data/Output/ChatGPTResults/chatgpt_lime_explanation.html
